# 01 — Làm sạch dữ liệu tiền điện tử

## Mục tiêu và nguyên tắc xử lý

Notebook đọc từng CSV trong kho nén Coin Metrics để chuẩn hóa ngày, mã tài sản và các trường số. Dữ liệu sau làm sạch được tách thành bảng phục vụ phân tích giá và bảng phục vụ phân tích thị trường.

- Khóa của mỗi bảng là `(ticker, date)`. Ngày được lấy từ cột `time`, còn `ticker` được lấy từ tên CSV thành viên và chuyển thành chữ hoa.
- Một dòng chỉ vào bảng giá khi có ngày hợp lệ và `PriceUSD > 0`. Giá thiếu không được nội suy.
- `CapMrktCurUSD` và `CapMrktEstUSD` được giữ thành hai trường riêng. Vốn hóa theo nguồn cung hiện tại là cơ sở cho xếp hạng, thị phần và tỷ trọng vốn hóa BTC ở các bước sau.
- `volume_reported_spot_usd_1d` được đổi tên thành `reported_spot_volume_usd`, thể hiện khối lượng giao dịch spot do nguồn báo cáo. Đây là trường khác với giá trị chuyển tiền trên chuỗi.
- Giá trị volume lớn hơn `10^15 USD/ngày` được gắn cờ và chuyển thành thiếu trong bảng xử lý. Ngưỡng này là quy tắc kiểm tra của dự án; snapshot đang dùng được kiểm tra có đúng ba dòng bị gắn cờ.
- Các chỉ số on-chain được giữ khi nguồn cung cấp, không dùng để thay thế giá hoặc vốn hóa còn thiếu. Các tệp dữ liệu thô được giữ nguyên.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd() / "scripts"))
from common import STAGING, TABLES, QUALITY, ensure_output_dirs, iter_coinmetrics, assert_unique_key, write_csv_atomic
ensure_output_dirs()

## Phân nhóm tài sản để mô tả

Bảng `CATEGORY` gán nhóm thủ công cho một số mã tài sản, chẳng hạn Layer 1, Stablecoin, DeFi hoặc Privacy. Mã không có trong danh sách sẽ được ghi là `Unclassified` khi tạo `dim_coin`.

Các nhóm này giúp tổ chức và so sánh dữ liệu trong dự án. Chúng không phải một hệ phân loại được cập nhật tự động và không đủ để giải thích hiệu quả đầu tư hay quan hệ nhân quả.

In [2]:
CATEGORY = {
    "BTC":"Layer 1","ETH":"Layer 1","BNB":"Layer 1","ADA":"Layer 1","SOL":"Layer 1",
    "XRP":"Payment","DOGE":"Meme","SHIB":"Meme","DOT":"Layer 1","AVAX":"Layer 1",
    "TRX":"Layer 1","LTC":"Payment","BCH":"Payment","XLM":"Payment","ETC":"Layer 1",
    "XMR":"Privacy","ZEC":"Privacy","DASH":"Privacy","ATOM":"Layer 1","NEAR":"Layer 1",
    "ALGO":"Layer 1","VET":"Layer 1","HBAR":"Layer 1","EOS":"Layer 1","NEO":"Layer 1",
    "QTUM":"Layer 1","XTZ":"Layer 1","ICP":"Layer 1","FTM":"Layer 1","THETA":"Layer 1",
    "WAVES":"Layer 1","APT":"Layer 1","MATIC":"Layer 2","LRC":"Layer 2",
    "USDT":"Stablecoin","USDC":"Stablecoin","UNI":"DeFi","AAVE":"DeFi","MKR":"DeFi",
    "COMP":"DeFi","CRV":"DeFi","SNX":"DeFi","YFI":"DeFi","SUSHI":"DeFi",
    "1INCH":"DeFi","ZRX":"DeFi","KNC":"DeFi","REN":"DeFi","BAT":"Utility",
    "LINK":"Oracle","GRT":"Infrastructure","FIL":"Storage","STORJ":"Storage",
    "CHZ":"Gaming","GALA":"Gaming","AXS":"Gaming","SAND":"Metaverse","MANA":"Metaverse",
    "ENJ":"Gaming","ANKR":"Infrastructure","REP":"DeFi","OMG":"Layer 2","EGLD":"Layer 1",
}

## Chuẩn hóa trường dữ liệu và tách hai bảng trung gian

Ngày hoặc giá trị số không chuyển đổi được sẽ trở thành giá trị thiếu. Giá, hai loại vốn hóa, nguồn cung và MVRV chỉ được giữ khi dương; volume, số lượng địa chỉ/giao dịch, hashrate và dòng tiền ra vào sàn chỉ được giữ khi không âm. Nếu một CSV không có trường cần chọn, cột tương ứng vẫn được tạo và để thiếu.

Hai bảng trung gian có điều kiện giữ dòng khác nhau:

| Bảng | Điều kiện giữ dòng | Nội dung chính |
|---|---|---|
| `crypto_base` | Ngày hợp lệ và giá USD dương | Giá, các trường thị trường, chỉ số on-chain và cờ volume bất thường |
| `market_clean` | Ngày hợp lệ và có ít nhất một giá trị vốn hóa hoặc volume hợp lệ | Hai loại vốn hóa, volume spot và cờ volume bất thường |

Nhờ cách tách này, dòng thiếu giá vẫn có thể đóng góp vào phân tích thị trường nếu còn vốn hóa hoặc volume hợp lệ. Sau khi ghép dữ liệu của các tài sản, notebook kiểm tra khóa `(ticker, date)` và các miền giá trị. Báo cáo làm sạch ghi số dòng thiếu hoặc không hợp lệ theo từng tài sản; các số đếm này không đồng nghĩa với số dòng bị loại khỏi cả hai bảng.

In [3]:
RAW_TO_CLEAN = {
    "PriceUSD": "price_usd",
    "CapMrktCurUSD": "market_cap_current_usd",
    "CapMrktEstUSD": "market_cap_estimated_usd",
    "volume_reported_spot_usd_1d": "reported_spot_volume_usd",
    "AdrActCnt": "active_addresses",
    "TxCnt": "tx_count",
    "TxTfrCnt": "transfer_count",
    "HashRate": "hashrate",
    "SplyCur": "supply_current_native",
    "CapMVRVCur": "mvrv_current",
    "FlowInExUSD": "exchange_inflow_usd",
    "FlowOutExUSD": "exchange_outflow_usd",
}

price_frames, market_frames, metadata_rows, audit_rows = [], [], [], []
for ticker, source in iter_coinmetrics():
    frame = source.copy()
    frame["date"] = pd.to_datetime(frame["time"], errors="coerce").dt.normalize()
    for raw_column, clean_column in RAW_TO_CLEAN.items():
        frame[clean_column] = (
            pd.to_numeric(frame[raw_column], errors="coerce")
            if raw_column in frame.columns
            else np.nan
        )

    strictly_positive = [
        "price_usd", "market_cap_current_usd", "market_cap_estimated_usd",
        "supply_current_native", "mvrv_current",
    ]
    nonnegative = [
        "reported_spot_volume_usd", "active_addresses", "tx_count",
        "transfer_count", "hashrate", "exchange_inflow_usd", "exchange_outflow_usd",
    ]
    for column in strictly_positive:
        frame[column] = frame[column].where(frame[column] > 0)
    for column in nonnegative:
        frame[column] = frame[column].where(frame[column] >= 0)

    # Project validity rule for an obvious source-magnitude failure. The threshold is
    # intentionally many orders above plausible global daily spot turnover; raw stays immutable.
    volume_invalid_threshold_usd_per_day = 1e15
    frame["reported_spot_volume_invalid_flag"] = (
        frame["reported_spot_volume_usd"].gt(volume_invalid_threshold_usd_per_day)
    )
    frame.loc[
        frame["reported_spot_volume_invalid_flag"], "reported_spot_volume_usd"
    ] = np.nan

    common = frame[["date", *RAW_TO_CLEAN.values(), "reported_spot_volume_invalid_flag"]].copy()
    common.insert(1, "ticker", ticker.upper())
    valid_dates = common["date"].notna()
    dated = common.loc[valid_dates].copy()

    price_panel = dated.dropna(subset=["price_usd"]).copy()
    market_panel = dated.dropna(
        subset=[
            "market_cap_current_usd",
            "market_cap_estimated_usd",
            "reported_spot_volume_usd",
        ],
        how="all",
    ).copy()
    price_frames.append(price_panel)
    market_frames.append(
        market_panel[
            [
                "date", "ticker", "market_cap_current_usd",
                "market_cap_estimated_usd", "reported_spot_volume_usd",
                "reported_spot_volume_invalid_flag",
            ]
        ]
    )

    metadata_rows.append(
        {
            "ticker": ticker.upper(),
            "raw_first_date": dated["date"].min(),
            "raw_last_date": dated["date"].max(),
            "raw_rows": len(source),
            "price_first_date": price_panel["date"].min(),
            "price_last_date": price_panel["date"].max(),
            "valid_price_rows": len(price_panel),
            "valid_market_rows": len(market_panel),
            "valid_current_cap_rows": int(market_panel["market_cap_current_usd"].notna().sum()),
            "valid_estimated_cap_rows": int(market_panel["market_cap_estimated_usd"].notna().sum()),
        }
    )
    audit_rows.append(
        {
            "ticker": ticker.upper(),
            "raw_rows": len(source),
            "invalid_date_rows": int(frame["date"].isna().sum()),
            "invalid_or_missing_price_rows": int(frame["price_usd"].isna().sum()),
            "invalid_or_missing_current_cap_rows": int(frame["market_cap_current_usd"].isna().sum()),
            "invalid_or_missing_estimated_cap_rows": int(frame["market_cap_estimated_usd"].isna().sum()),
            "invalid_or_missing_reported_spot_volume_rows": int(frame["reported_spot_volume_usd"].isna().sum()),
            "invalid_reported_spot_volume_magnitude_rows": int(frame["reported_spot_volume_invalid_flag"].sum()),
            "action": "invalid numeric values set missing; >1e15 USD/day reported volume flagged and set missing; price rows excluded only from price panel; no imputation",
        }
    )

crypto_base = pd.concat(price_frames, ignore_index=True).sort_values(["ticker", "date"])
market_clean = pd.concat(market_frames, ignore_index=True).sort_values(["ticker", "date"])
metadata = pd.DataFrame(metadata_rows)
cleaning_audit = pd.DataFrame(audit_rows)

assert_unique_key(crypto_base, ["ticker", "date"], "crypto_base")
assert_unique_key(market_clean, ["ticker", "date"], "market_clean")
assert (crypto_base["price_usd"] > 0).all()
assert market_clean["market_cap_current_usd"].dropna().gt(0).all()
assert market_clean["market_cap_estimated_usd"].dropna().gt(0).all()
assert market_clean["reported_spot_volume_usd"].dropna().between(0, 1e15, inclusive="both").all()
assert int(market_clean["reported_spot_volume_invalid_flag"].sum()) == 3

display(crypto_base.head())
display(cleaning_audit.head())

,date,ticker,price_usd,market_cap_current_usd,market_cap_estimated_usd,reported_spot_volume_usd,active_addresses,tx_count,transfer_count,hashrate,supply_current_native,mvrv_current,exchange_inflow_usd,exchange_outflow_usd,reported_spot_volume_invalid_flag
0,2020-12-26,1INCH,1.619328,2.428992e+09,1.053593e+08,1.502920e+08,12305.0,27081.0,33665.0,NaN,1.500000e+09,17.963154,NaN,NaN,False
1,2020-12-27,1INCH,1.072350,1.608525e+09,7.995606e+07,1.329511e+08,6016.0,13526.0,17187.0,NaN,1.500000e+09,16.880111,NaN,NaN,False
2,2020-12-28,1INCH,1.104674,1.657011e+09,7.971789e+07,5.989677e+07,3929.0,7700.0,9924.0,NaN,1.500000e+09,16.574084,NaN,NaN,False
3,2020-12-29,1INCH,0.893790,1.340686e+09,6.682589e+07,6.162150e+07,3330.0,7326.0,10132.0,NaN,1.500000e+09,16.320919,NaN,NaN,False
4,2020-12-30,1INCH,1.101307,1.651960e+09,8.243729e+07,8.259668e+07,2581.0,6347.0,8394.0,NaN,1.500000e+09,16.639024,NaN,NaN,False


,ticker,raw_rows,invalid_date_rows,invalid_or_missing_price_rows,invalid_or_missing_current_cap_rows,invalid_or_missing_estimated_cap_rows,invalid_or_missing_reported_spot_volume_rows,invalid_reported_spot_volume_magnitude_rows,action
0,1INCH,1979,0,4,4,4,3,0,invalid numeric values set missing; >1e15 USD/...
1,AAVE,2069,0,17,17,17,9,0,invalid numeric values set missing; >1e15 USD/...
2,ADA,3166,0,70,70,638,70,0,invalid numeric values set missing; >1e15 USD/...
3,ALGO,2540,0,12,12,12,12,0,invalid numeric values set missing; >1e15 USD/...
4,ANKR,2627,0,2627,2627,99,1,0,invalid numeric values set missing; >1e15 USD/...


## Tạo bảng thông tin tài sản, xếp hạng và lưu kết quả

`dim_coin` chứa một dòng cho mỗi `ticker`, gồm khoảng thời gian trong nguồn, khoảng thời gian có giá hợp lệ, số dòng có dữ liệu thị trường và nhóm mô tả.

Xếp hạng vốn hóa được tính tại một ngày chung: ngày gần nhất trong `market_clean` có vốn hóa hiện tại của ít nhất một tài sản. Chỉ những tài sản có `market_cap_current_usd` tại đúng ngày đó được xếp hạng. Tài sản thiếu giá trị tại ngày chung được giữ trong `dim_coin` nhưng để thiếu vốn hóa và thứ hạng; không dùng vốn hóa từ ngày cũ hoặc vốn hóa ước tính để bù.

Notebook lưu `crypto_base.csv.gz` và `market_clean.csv.gz` vào `data/processed/staging/`, `dim_coin.csv` vào `data/processed/tables/`, và báo cáo `crypto_cleaning_audit.csv` vào `data/processed/quality_reports/`.

In [4]:
dim_coin = metadata.copy()
dim_coin["has_price_history"] = dim_coin["valid_price_rows"].gt(0)
dim_coin["category"] = dim_coin["ticker"].map(CATEGORY).fillna("Unclassified")
dim_coin["category_source"] = "manual_project_taxonomy_descriptive_only"

as_of_date = market_clean.loc[market_clean["market_cap_current_usd"].notna(), "date"].max()
latest_current = (
    market_clean.loc[
        market_clean["date"].eq(as_of_date) & market_clean["market_cap_current_usd"].notna(),
        ["ticker", "market_cap_current_usd"],
    ]
    .rename(columns={"market_cap_current_usd": "market_cap_current_as_of_usd"})
    .sort_values("market_cap_current_as_of_usd", ascending=False)
)
latest_current["market_cap_current_rank_as_of"] = (
    latest_current["market_cap_current_as_of_usd"]
    .rank(method="min", ascending=False)
    .astype("Int64")
)
latest_current["market_cap_as_of_date"] = as_of_date
dim_coin = dim_coin.merge(latest_current, on="ticker", how="left", validate="one_to_one")
dim_coin = dim_coin.sort_values("ticker")
assert_unique_key(dim_coin, ["ticker"], "dim_coin")

write_csv_atomic(crypto_base, STAGING / "crypto_base.csv.gz")
write_csv_atomic(market_clean, STAGING / "market_clean.csv.gz")
write_csv_atomic(dim_coin, TABLES / "dim_coin.csv")
write_csv_atomic(cleaning_audit, QUALITY / "crypto_cleaning_audit.csv")

print(f"Price panel: {len(crypto_base):,} rows, {crypto_base.ticker.nunique()} tickers")
print(f"Market panel: {len(market_clean):,} rows, {market_clean.ticker.nunique()} tickers")
print(f"Common current-cap rank date: {as_of_date:%Y-%m-%d}")
display(dim_coin.sort_values("market_cap_current_rank_as_of").head(15))

Price panel: 123,358 rows, 39 tickers
Market panel: 185,694 rows, 63 tickers
Common current-cap rank date: 2026-05-23


,ticker,raw_first_date,raw_last_date,raw_rows,price_first_date,price_last_date,valid_price_rows,valid_market_rows,valid_current_cap_rows,valid_estimated_cap_rows,has_price_history,category,category_source,market_cap_current_as_of_usd,market_cap_current_rank_as_of,market_cap_as_of_date
12,BTC,2009-01-03,2026-05-24,6351,2010-07-18,2026-05-23,5789,5789,5789,2528,True,Layer 1,manual_project_taxonomy_descriptive_only,1.534955e+12,1,2026-05-23
23,ETH,2015-07-30,2026-05-24,3952,2015-08-08,2026-05-23,3942,3942,3942,2528,True,Layer 1,manual_project_taxonomy_descriptive_only,2.575830e+11,2,2026-05-23
53,USDT,2013-12-28,2026-05-24,4531,2014-10-06,2026-05-23,4248,4530,4248,2528,True,Stablecoin,manual_project_taxonomy_descriptive_only,1.927645e+11,3,2026-05-23
58,XRP,2013-01-01,2026-05-24,4892,2014-08-15,2026-05-23,4300,4762,4300,2528,True,Payment,manual_project_taxonomy_descriptive_only,1.357630e+11,4,2026-05-23
52,USDC,2018-08-03,2026-05-24,2852,2018-09-28,2026-05-23,2795,2795,2795,2528,True,Stablecoin,manual_project_taxonomy_descriptive_only,7.254175e+10,5,2026-05-23
17,DOGE,2013-12-08,2026-05-24,4551,2014-01-23,2026-05-23,4504,4505,4504,2528,True,Meme,manual_project_taxonomy_descriptive_only,1.589589e+10,6,2026-05-23
56,XLM,2014-08-12,2026-05-24,4304,2015-09-30,2026-05-23,3889,4303,3889,2528,True,Payment,manual_project_taxonomy_descriptive_only,1.548035e+10,7,2026-05-23
61,ZEC,2016-10-28,2026-05-24,3496,2016-10-29,2026-05-23,3494,3494,3494,2528,True,Privacy,manual_project_taxonomy_descriptive_only,1.058596e+10,8,2026-05-23
31,LINK,2017-09-16,2026-05-24,3173,2017-09-29,2026-05-23,3159,3159,3159,2528,True,Oracle,manual_project_taxonomy_descriptive_only,9.556262e+09,9,2026-05-23
2,ADA,2017-09-23,2026-05-24,3166,2017-12-01,2026-05-23,3096,3096,3096,2528,True,Layer 1,manual_project_taxonomy_descriptive_only,8.981194e+09,10,2026-05-23
